In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv('/content/drive/MyDrive/realty_data.csv',sep=',')
df.head()

Mounted at /content/drive


,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [16]:
!pip install fastapi uvicorn scikit-learn nest-asyncio pyngrok

In [7]:
df = df[['total_square', 'floor', 'price']].dropna()

X = df[['total_square', 'floor']]
y = df['price']

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)


LinearRegression()

In [8]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok

app = FastAPI()

class Item(BaseModel):
    total_square: float
    floor: float

@app.get("/health")
def health_check():
    return {"status": "alive"}

@app.get("/predict_get")
def predict_get(total_square: float, floor: float):
    prediction = model.predict([[total_square, floor]])
    return {"predicted_price": prediction[0]}

@app.post("/predict_post")
def predict_post(item: Item):
    prediction = model.predict([[item.total_square, item.floor]])
    return {"predicted_price": prediction[0]}


In [10]:
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-386.tgz

--2025-04-27 08:54:36--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-386.tgz
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 75.2.60.68, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8873140 (8.5M) [application/octet-stream]
Saving to: ‘ngrok-v3-stable-linux-386.tgz’

ngrok-v3-stable-lin 100%[===================>]   8.46M  27.1MB/s    in 0.3s    

2025-04-27 08:54:37 (27.1 MB/s) - ‘ngrok-v3-stable-linux-386.tgz’ saved [8873140/8873140]



In [11]:
!tar -xvf ngrok-v3-stable-linux-386.tgz

ngrok


In [12]:
!./ngrok config add-authtoken 2vzIDbp01iqhvgWxedSusAfl8m9_Yqb4W7VhEVpoSJoE3Tbp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [13]:
!pip install pyngrok

In [ ]:
!python -m uvicorn app --reload --host 0.0.0.0 --port 8501

In [15]:
nest_asyncio.apply()


public_url = ngrok.connect(8000)
print(f"Сервер запущен! Swagger документация доступна здесь: {public_url}/docs")

uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [581]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Сервер запущен! Swagger документация доступна здесь: NgrokTunnel: "https://287d-34-41-232-255.ngrok-free.app" -> "http://localhost:8000"/docs
INFO:     57.129.20.201:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     57.129.20.201:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     57.129.20.201:0 - "GET /health HTTP/1.1" 200 OK
INFO:     57.129.20.201:0 - "GET /predict_get HTTP/1.1" 422 Unprocessable Entity
INFO:     57.129.20.201:0 - "GET /predict_post HTTP/1.1" 405 Method Not Allowed


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [581]
